In [3]:
import numpy as np
import pandas as pd

In [4]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer

In [5]:
df = pd.read_csv('Titanic-Dataset.csv',usecols=['Age','Fare','SibSp','Parch','Survived'])

In [6]:
df.head()

,Survived,Age,SibSp,Parch,Fare
0,0,22.0,1,0,7.2500
1,1,38.0,1,0,71.2833
2,1,26.0,0,0,7.9250
3,1,35.0,1,0,53.1000
4,0,35.0,0,0,8.0500


In [7]:
#Age has missing values
df.isnull().sum()

Survived      0
Age         177
SibSp         0
Parch         0
Fare          0
dtype: int64

In [8]:
df.dropna(inplace=True)

In [9]:
df.head()

,Survived,Age,SibSp,Parch,Fare
0,0,22.0,1,0,7.2500
1,1,38.0,1,0,71.2833
2,1,26.0,0,0,7.9250
3,1,35.0,1,0,53.1000
4,0,35.0,0,0,8.0500


In [10]:
df['family'] = df['SibSp'] + df['Parch']

In [11]:
df.head()

,Survived,Age,SibSp,Parch,Fare,family
0,0,22.0,1,0,7.2500,1
1,1,38.0,1,0,71.2833,1
2,1,26.0,0,0,7.9250,0
3,1,35.0,1,0,53.1000,1
4,0,35.0,0,0,8.0500,0


In [12]:
df.drop(columns=['SibSp','Parch'],inplace=True)

In [13]:
df.head()

,Survived,Age,Fare,family
0,0,22.0,7.2500,1
1,1,38.0,71.2833,1
2,1,26.0,7.9250,0
3,1,35.0,53.1000,1
4,0,35.0,8.0500,0


In [14]:
#now we take X and y values
X = df.drop(columns=['Survived'])
y = df['Survived']

In [15]:
# Now we do train test split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [16]:
X_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


In [17]:
X_test.head()

,Age,Fare,family
149,42.0,13.00,0
407,3.0,18.75,2
53,29.0,26.00,1
369,24.0,69.30,0
818,43.0,6.45,0


In [18]:
y_train.head()

328    1
73     0
253    0
719    0
666    0
Name: Survived, dtype: int64

In [19]:
y_test.head()

149    0
407    1
53     1
369    1
818    0
Name: Survived, dtype: int64

In [20]:
# Without Binarization
clf1 = DecisionTreeClassifier()
clf1.fit(X_train,y_train)
y_pred1 = clf1.predict(X_test)

accuracy_score(y_test,y_pred1)

0.6293706293706294

In [21]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6499413145539906)

In [22]:
#Apply Binarization
#first make a column transformer
from sklearn.preprocessing import Binarizer

In [28]:
trf = ColumnTransformer([
    ('Bin', Binarizer(copy=False), ['family'])
], remainder='passthrough', sparse_threshold=0)

In [29]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.fit_transform(X_test)

In [ ]:
# ...existing code...
pd.DataFrame(X_train_trf, columns=['family', 'Age', 'Fare'])
# ...existing code...

,family,Age,Fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [31]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test,y_pred2)

0.6223776223776224

In [32]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X_trf,y,cv=10,scoring='accuracy'))

np.float64(0.623356807511737)